# Week 5 · 07 Paper practice: deep learning by hand, in code, and in PyTorch

Exam-style practice for the whole week, in the three forms the lab exam uses: computing by hand, writing code by hand, and reading PyTorch code. Part IV mixes in Week 4 so it doesn't fade. About 75 minutes on paper, then type Part II's functions and run the checks.

**How to use this notebook**
- Run cells one at a time with **Shift + Enter**.
- Before running an example, **guess what it prints**. The real result is written after `# →` in the comments, so you can check yourself.
- **Your turn** cells are for you. Open **Show one answer** only after a real attempt.
- If things get weird, click **Restart** at the top and run from the top again.

**Run this cell first.** It loads NumPy, the picture helpers (`figures.py` in this folder) and the exercise checker.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from figures import *
import checks; checks.load(globals(), "07")      # defines check(n)

## M. How to use this notebook

Set a timer for 75 minutes and work on paper: Part I (by hand, about 35 minutes), Part II (code by hand, about 25 minutes), Part III (reading PyTorch, about 15 minutes). Then mark yourself with the worked answers, type Part II into the cells and run the checks. Part IV (Week 4, spaced) is extra. Log the score and every miss **by topic** in the tracker.

| Part | What | Points |
|---|---|---|
| I | by hand: a forward and backward pass, graphs, optimizers, proofs | 8 × 5 = 40 |
| II | handwritten code, then checked | 5 × 8 = 40 |
| III | reading PyTorch code | 5 × 4 = 20 |
| IV | Week 4, spaced (not scored) |  |

## I. By hand (40 points)

The network for I1–I2: 2 inputs, 3 ReLU hidden units, 2 classes with softmax and cross-entropy. $x = (1, 2)$, true class $y = 1$.

$$
W_1 = \begin{pmatrix} 1 & -1 & 0 \\ 0.5 & 1 & -1 \end{pmatrix},\quad b_1 = (0, 0, 1),\quad W_2 = \begin{pmatrix} 1 & 0 \\ 0 & 1 \\ 1 & 1 \end{pmatrix},\quad b_2 = (0, 0)
$$

**Drill (on paper).** **I1 · Forward.** Compute $z_1 = xW_1 + b_1$, $h = \text{ReLU}(z_1)$, the logits $z_2 = hW_2 + b_2$, the probabilities, and the loss.

<details><summary>Worked answer</summary>

$z_1 = (1 \cdot 1 + 2 \cdot 0.5,\ -1 + 2,\ 0 - 2 + 1) = (2, 1, -1)$; $h = (2, 1, 0)$; $z_2 = (2 + 0 + 0,\ 0 + 1 + 0) = (2, 1)$; $p = \operatorname{softmax}(2, 1) = (0.731, 0.269)$; $L = -\log 0.269 = 1.313$.

</details>

**Drill (on paper).** **I2 · Backward.** Compute $\partial L / \partial z_2$, $\partial L / \partial W_2$, $\partial L / \partial h$, $\partial L / \partial z_1$ and $\partial L / \partial W_1$.

<details><summary>Worked answer</summary>

$\partial L / \partial z_2 = p - e_1 = (0.731, -0.731)$. $\ \partial L / \partial W_2 = h^\top (p - e_1) = \begin{pmatrix} 1.462 & -1.462 \\ 0.731 & -0.731 \\ 0 & 0 \end{pmatrix}$ (a row per hidden unit: $h_k \times$ the gradient). $\ \partial L / \partial h = (p - e_1) W_2^\top = (0.731, -0.731, 0)$. $\ \partial L / \partial z_1 = $ that $\odot\, \mathbb{1}[z_1 > 0] = (0.731, -0.731, 0)$. $\ \partial L / \partial W_1 = x^\top (\partial L / \partial z_1) = \begin{pmatrix} 0.731 & -0.731 & 0 \\ 1.462 & -1.462 & 0 \end{pmatrix}$. The third hidden unit was off ($z = -1$), so nothing flows through it: its column is 0.

</details>

**Drill (on paper).** **I3 · Counting.** A network $100 \to 64 \to 64 \to 5$. How many parameters? What's the output shape for a batch of 32?

<details><summary>Worked answer</summary>

$100 \cdot 64 + 64 = 6464$; $64 \cdot 64 + 64 = 4160$; $64 \cdot 5 + 5 = 325$; total **10,949**. Output `(32, 5)`.

</details>

**Drill (on paper).** **I4 · Optimizers.** (a) Momentum with $\eta = 0.1$, $\beta = 0.9$: the velocity is $v = 1$, the weight $w = 1$, the new gradient $g = 2$. New $v$ and $w$? (b) Adam at its first step ($t = 1$, $m = s = 0$) with $\eta = 0.01$ and $g = -40$: how much does $w$ change, and why doesn't the size of $g$ matter?

<details><summary>Worked answer</summary>

(a) $v = 0.9 \cdot 1 + 2 = 2.9$; $w = 1 - 0.1 \cdot 2.9 = 0.71$. (b) $m = -4$, $s = 1.6$; $\hat m = -4 / 0.1 = -40$, $\hat s = 1.6 / 0.001 = 1600$; step $= 0.01 \cdot (-40) / \sqrt{1600} = -0.01$, so $w$ **increases** by $0.01$. At $t = 1$, $\hat m / \sqrt{\hat s} = g / \lvert g \rvert = \pm 1$ for any $g$: the first step always has size $\eta$.

</details>

**Drill (on paper).** **I5 · A graph.** $f = (a b + c)^2$ with $a = 2$, $b = -1$, $c = 4$. Forward values, then $\partial f / \partial a$, $\partial f / \partial b$, $\partial f / \partial c$.

<details><summary>Worked answer</summary>

$u = ab = -2$, $q = u + c = 2$, $f = q^2 = 4$. Backward: $\partial f / \partial q = 2q = 4$; the add passes it on: $\partial f / \partial u = \partial f / \partial c = 4$; the multiply swaps: $\partial f / \partial a = 4 \cdot b = -4$, $\partial f / \partial b = 4 \cdot a = 8$.

</details>

**Drill (on paper).** **I6 · Vanishing gradients.** A chain of 6 sigmoid units, every weight of size at most 1. Give an upper bound for the factor by which the gradient shrinks from the last unit to the first, and name two remedies.

<details><summary>Worked answer</summary>

Each factor is $\lvert w \rvert\, \sigma'(z) \le 1 \cdot \frac{1}{4}$, so the bound is $(1/4)^6 \approx 2.4 \times 10^{-4}$. Remedies: ReLU activations (derivative 1 when active), He/Xavier initialization, batch normalization, residual connections.

</details>

**Drill (on paper).** **I7 · A proof.** Show that a network made only of linear layers (no activations) can't classify XOR.

<details><summary>Worked answer</summary>

Two linear layers compose to one: $(xW_1 + b_1)W_2 + b_2 = x(W_1W_2) + (b_1W_2 + b_2)$, and by induction any number of them is a single linear layer $xW + b$, whose decision boundary is a line. No line separates XOR: $b < 0$, $w_1 + w_2 + b < 0$, $w_1 + b \ge 0$, $w_2 + b \ge 0$ would give $w_1 + w_2 + 2b$ both $\ge 0$ and $< 0$.

</details>

**Drill (on paper).** **I8 · Softmax gradient.** Logits $(1, 2, 3)$, true class $y = 1$. Give $p$ and $\partial L / \partial z$ for the cross-entropy loss.

<details><summary>Worked answer</summary>

$p = (0.090, 0.245, 0.665)$; $\partial L / \partial z = p - e_1 = (0.090, -0.755, 0.665)$. The true class's logit gets the only negative gradient, so gradient descent raises it.

</details>

## II. Handwritten code (40 points)

Write each function on paper first. Then type it in and run its check. Exercise 1 and 4 are from the rebuild list (`00` §C).

**Exercise 1 · k-means, from memory.** Starting from centroids `C` `(k, d)`, repeat `iters` times: assign every row of `X` `(n, d)` to its nearest centroid, then move each centroid to the mean of its rows. Return the final centroids, **shape `(k, d)`**, without changing the `C` passed in. (The tests never leave a cluster empty.)

**Example**: what goes in, and what should come out:
```python
X = np.array([[1.0, 1.0], [1.0, 2.0], [2.0, 1.0], [4.0, 3.0], [5.0, 4.0], [5.0, 3.0]])
kmeans_fit(X, np.array([[1.0, 1.0], [2.0, 1.0]]), 2)
# → [[1.333 1.333]
#    [4.667 3.333]]     Week 4 · 03's two iterations
```

In [31]:
X = np.array([[1.0, 1.0], [1.0, 2.0], [2.0, 1.0], [4.0, 3.0], [5.0, 4.0], [5.0, 3.0]])
C = np.array([[1.0, 1.0], [2.0, 1.0]])
print(X.shape)
print(((X[:, None, :] - C[None, :, :]) ** 2).sum(axis=2).argmin(axis=1))
for row in X:
    idx = np.argmin(((C-row)**2).sum(axis=1))
    break
    print(idx)

(6, 2)
[0 0 1 1 1 1]


In [29]:
def kmeans_fit(X, C, iters):
    C = C.copy()                                               # don't change the caller's centroids
    for _ in range(iters):
        labels = ((X[:, None, :] - C[None, :, :]) ** 2).sum(axis=2).argmin(axis=1)   # (n,)
        C = np.array([X[labels == j].mean(axis=0) for j in range(len(C))])         # (k, d)
    return C

In [30]:
check(1)

Exercise 1: PASS


**Exercise 2 · Softmax cross-entropy, forward and backward.** Return `(loss, dZ)`: the mean cross-entropy of the softmax of `Z` `(n, C)` against integer labels `y` `(n,)`, and its gradient $\frac{1}{n}(P - Y)$, **shape `(n, C)`**.

**Example**: what goes in, and what should come out:
```python
loss, dZ = softmax_cross_entropy(np.array([[2.0, 1.0]]), np.array([1]))
loss    # → 1.3133
dZ      # → [[ 0.731 -0.731]]     I1–I2's numbers
```

In [44]:
def softmax_cross_entropy(Z, y):
    E = np.exp(Z - Z.max(axis=1, keepdims=True))
    P = E / E.sum(axis=1, keepdims=True)
    n = len(y)
    loss = -np.log(P[np.arange(n), y]).mean() # softmax cross-entropy loss..
    dZ = (P - np.eye(Z.shape[1])[y]) / n
    return loss, dZ

In [45]:
check(2)

Exercise 2: PASS


For one example with logits $z = (z_1, \dots, z_C)$ and correct class $y$:

$$\ell(z, y) = -\log \frac{e^{z_y}}{\sum_{c=1}^{C} e^{z_c}} = -z_y + \log \sum_{c=1}^{C} e^{z_c}$$

For a batch of $n$ examples, it's the average:

$$L = \frac{1}{n}\sum_{i=1}^{n} \left(-z_{i,y_i} + \log \sum_{c=1}^{C} e^{z_{ic}}\right)$$

**Exercise 3 · One training step of a two-layer MLP.** ReLU hidden layer, softmax output, mean cross-entropy. `X` `(n, d)`, `y` `(n,)`, `W1` `(d, h)`, `b1` `(h,)`, `W2` `(h, C)`, `b2` `(C,)`. Forward, backward, one gradient descent step with `lr`. Return the new `(W1, b1, W2, b2)` as **new** arrays; don't change the inputs.

**Example**: what goes in, and what should come out:
```python
W1 = np.array([[1, -1, 0], [0.5, 1, -1.0]]); b1 = np.array([0, 0, 1.0])
W2 = np.array([[1, 0], [0, 1], [1, 1.0]]); b2 = np.zeros(2)
new = mlp_train_step(np.array([[1.0, 2.0]]), np.array([1]), W1, b1, W2, b2, 0.1)
new[2]    # → [[ 0.854  0.146]
          #    [-0.073  1.073]
          #    [ 1.     1.   ]]     W2 - 0.1 × I2's gradient
```

In [46]:
def mlp_train_step(X, y, W1, b1, W2, b2, lr):
    Z1 = X @ W1 + b1
    H = np.maximum(Z1, 0)
    _, G = softmax_cross_entropy(H @ W2 + b2, y)
    dW2, db2 = H.T @ G, G.sum(axis=0)
    G1 = (G @ W2.T) * (Z1 > 0)
    dW1, db1 = X.T @ G1, G1.sum(axis=0)
    return W1 - lr * dW1, b1 - lr * db1, W2 - lr * dW2, b2 - lr * db2

In [47]:
check(3)

Exercise 3: PASS


**Exercise 4 · Pairwise squared distances, from memory.** $D_{ij} = \sum_d (A_{id} - B_{jd})^2$ for `A` `(m, d)` and `B` `(p, d)`. Return **shape `(m, p)`**, no loops.

**Example**: what goes in, and what should come out:
```python
pairwise_sq_dists(np.array([[0.0, 0.0]]), np.array([[3.0, 4.0], [1.0, 0.0]]))    # → [[25.  1.]]
```

In [ ]:
def pairwise_sq_dists(A, B):
    return ((A[:,None,:] - B[None,:,:])**2).sum(axis=2)
    #마지가 axis 그냥 없는걸로 생각하고, A[:,None] --> row로 세우고, B[None,:] --> column으로 A에다가 broadcasting 

In [49]:
check(4)

Exercise 4: PASS


**Exercise 5 · The PyTorch training loop, from memory.** Train `model` on all the data at once (`X` a float32 tensor `(n, d)`, `y` int64 `(n,)`) for `epochs` steps with `torch.optim.SGD(model.parameters(), lr=lr)` and `nn.CrossEntropyLoss()`. Return the list of the losses (`loss.item()`), one per step, each computed **before** that step's update.

**Example**: what goes in, and what should come out:
```python
torch.manual_seed(0)
net = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 2))
X_t = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]]); y_t = torch.tensor([0, 1, 1, 0])
losses = torch_fit(net, X_t, y_t, 200, 0.5)
losses[0] > losses[-1]     # → True   XOR, learned
```

In [50]:
def torch_fit(model, X, y, epochs, lr):
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    losses = []
    for _ in range(epochs):
        loss = loss_fn(model(X), y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return losses

In [51]:
check(5)

Exercise 5: PASS


In [52]:
for n in range(1, 6):
    check(n)

Exercise 1: PASS
Exercise 2: PASS
Exercise 3: PASS
Exercise 4: PASS
Exercise 5: PASS


## III. Reading PyTorch code (20 points)

**Drill (on paper).** **III1 · Shapes and parameters.**\n```python\nmodel = nn.Sequential(nn.Flatten(), nn.Linear(3 * 8 * 8, 50), nn.ReLU(), nn.Linear(50, 4))\nout = model(torch.randn(16, 3, 8, 8))\n```\nWhat's the shape after `nn.Flatten()`, the shape of `out`, and the number of parameters?

<details><summary>Worked answer</summary>

`nn.Flatten()` keeps the first (batch) dimension and joins the rest: `(16, 192)`. `out`: `(16, 4)`. Parameters: $192 \cdot 50 + 50 + 50 \cdot 4 + 4 = 9600 + 50 + 204 = 9854$.

</details>

**Drill (on paper).** **III2 · What does it print?**\n```python\nx = torch.ones(2, 3, requires_grad=True)\ny = (2 * x + 1).sum()\ny.backward()\nprint(y.item(), x.grad)\n```

<details><summary>Worked answer</summary>

`18.0` and `tensor([[2., 2., 2.], [2., 2., 2.]])`: $y = \sum (2x + 1) = 6 \cdot 3 = 18$, and $\partial y / \partial x_{ij} = 2$ for every entry. The gradient has $x$'s shape.

</details>

**Drill (on paper).** **III3 · What's wrong, and what happens?**\n```python\ntotal = 0\nfor xb, yb in loader:\n    loss = loss_fn(model(xb), yb)\n    opt.zero_grad(); loss.backward(); opt.step()\n    total += loss\n```

<details><summary>Worked answer</summary>

`total += loss` adds **tensors**, so `total` keeps a link to every batch's computation graph: memory grows over the epoch (and later, `total` itself would have a graph). Use `total += loss.item()` (a plain number).

</details>

**Drill (on paper).** **III4 · A subtle bug.**\n```python\nlogits = model(xb)           # (64, 10)\npred = torch.argmax(logits)\nacc = (pred == yb).float().mean()\n```

<details><summary>Worked answer</summary>

`torch.argmax(logits)` with no `dim` returns **one** index into the flattened `(640,)` tensor, not one prediction per row. It should be `logits.argmax(dim=1)`, shape `(64,)`. The comparison then broadcasts one number against 64 labels, so the code runs and gives a meaningless accuracy.

</details>

**Drill (on paper).** **III5 · Dropout at test time.** A model has `nn.Dropout(0.5)` layers, and the evaluation code forgets `model.eval()`. What do you observe?

<details><summary>Worked answer</summary>

Dropout stays on: half the hidden units are zeroed at random during evaluation, so the predictions are noisy and accuracy is lower than it should be, and it changes from run to run on the same test set.

</details>

## IV. Week 4, spaced (not scored)

**Drill (on paper).** **IV1 · One k-means iteration.** Points $(0, 0), (1, 0), (5, 5), (6, 5)$, starting centroids $(0, 0)$ and $(1, 0)$. Assign, then update.

<details><summary>Worked answer</summary>

Assign: $(0, 0) \to c_1$; $(1, 0) \to c_2$; $(5, 5)$: $d^2 = 50$ vs $41$ $\to c_2$; $(6, 5)$: $61$ vs $50$ $\to c_2$. Update: $c_1 = (0, 0)$, $c_2 = \left(\frac{1 + 5 + 6}{3}, \frac{0 + 5 + 5}{3}\right) = (4, 3.333)$. (A second iteration would move $(1, 0)$ to $c_1$.)

</details>

**Drill (on paper).** **IV2 · One responsibility.** Two Gaussians $\mathcal{N}(0, 1)$ and $\mathcal{N}(3, 1)$ with weights $\pi = (0.5, 0.5)$. The responsibility of component 1 for $x = 1$?

<details><summary>Worked answer</summary>

$\mathcal{N}(1 \mid 0, 1) \propto e^{-0.5} = 0.607$ and $\mathcal{N}(1 \mid 3, 1) \propto e^{-2} = 0.135$ (the $\frac{1}{\sqrt{2\pi}}$ and the equal $\pi$'s cancel). $r_1 = \frac{0.607}{0.607 + 0.135} = 0.818$.

</details>

---
Mark Parts I–III and log the score and misses by topic. Solutions for Part II: `solutions/07_solutions.py`. Next: `08_sunday_quiz.ipynb`.